[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 简单：交叉熵损失

从零实现**交叉熵损失**。

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

正确的位置 prob 越大, $\frac{e^{x_y}}{\sum_j e^{x_j}}$ 值越接近 1 , 交叉熵越小
### 函数签名
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) 浮点数, targets: (B,) 长整型索引
    # 返回: 标量损失（对批次取均值）
```

### 规则
- 禁止使用 `F.cross_entropy` 或 `nn.CrossEntropyLoss`
- 必须保持数值稳定性（使用 logsumexp 技巧）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

def cross_entropy_loss(logits, targets):
    pass  # log_probs = logits - logsumexp(...)

- 底数相乘 = 指数相加
$$a^m \cdot a^n = a^{m+n}$$
- 幂的乘方 = 指数相乘
$$(a^m)^n = a^{m \cdot n}$$

- 真数相乘 = 对数相加
$$\log_a(M \cdot N) = \log_a M + \log_a N$$
- 幂的对数 = 乘积
$$\log_a(M^n) = n \cdot \log_a M$$

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$
转换成
$$\text{CE}(x, y) = \log{\sum_j e^{x_j}} - \log{e^{x_y}}$$
转换成
$$ \text{CE}(x, y) = x_{max} + \log{\sum_j e^{x_j - x_{max}}} - \log{e^{x_y}}$$

In [ ]:
def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    实现数值稳定的交叉熵损失
    logits: (B, C) - 每个类别的未归一化得分
    targets: (B,) - 正确类别的索引
    """
    # 1. 找到每行的最大值，用于数值稳定性 (B, 1)
    max_logits, _ = torch.max(logits, dim=1, keepdim=True)
    
    # 2. 计算 Log-Sum-Exp: M + log(sum(exp(x - M)))
    # 减去 max_logits 后，指数部分最大为 0，有效防止溢出
    log_sum_exp = max_logits + torch.log(torch.sum(torch.exp(logits - max_logits), dim=1, keepdim=True))
    
    # 3. 计算 log_softmax: log_probs = x - log_sum_exp
    log_probs = logits - log_sum_exp
    
    # 4. 提取正确标签对应的 log_probs
    # 使用 torch.gather 或高级索引：log_probs[range(B), targets]
    batch_size = logits.shape[0]
    target_log_probs = log_probs[torch.arange(batch_size), targets]
    
    # 5. 计算负对数似然损失并取平均
    loss = -target_log_probs.mean()
    
    return loss

In [ ]:
# 🧪 调试
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('损失:', cross_entropy_loss(logits, targets))
print('参考:', torch.nn.functional.cross_entropy(logits, targets))

In [ ]:
# ✅ 提交
from torch_judge import check
check('cross_entropy')